In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import warnings
warnings.filterwarnings("ignore")

# Experimenting with DBSCAN and Hierarchical Clustering

In [ ]:
# DBSCAN Clustering — Census Income Segmentation

RANDOM_STATE = 42
SAMPLE_SIZE  = 20000  # increasing this causes memory issues

# data loading
def load_data(data_path, columns_path):
    with open(columns_path) as f:
        columns = [line.strip() for line in f if line.strip()]
    df = pd.read_csv(data_path, names=columns)
    return df

# data cleaning
def clean_data(df):
    df = df.replace("?", "Unknown")
    for col in ["detailed industry recode", "detailed occupation recode",
                "own business or self employed", "veterans benefits"]:
        df[col] = df[col].astype(str)

    for col in df.select_dtypes("object"):
        df[col] = df[col].str.strip()
    df["label"] = df["label"].str.replace(".", "", regex=False)

    df = df[df["age"] >= 18].reset_index(drop=True)
    df = df.drop_duplicates().reset_index(drop=True)
    return df

# feature engineering
def engineer_features(df):
    df["investment_activity"] = (
        (df["capital gains"] > 0) |
        (df["capital losses"] > 0) |
        (df["dividends from stocks"] > 0)
    ).astype(int)
    df["log_wage_per_hour"] = np.log1p(df["wage per hour"])
    df["log_capital_gains"] = np.log1p(df["capital gains"])
    df["log_dividends"] = np.log1p(df["dividends from stocks"])
    df["employment_stability"] = df["weeks worked in year"] * df["num persons worked for employer"]
    df["income_gt50k"] = df["label"].str.contains("50000\\+").astype(int)
    return df

# feature selection
def select_features(df):
    cols = [
        "age", "log_wage_per_hour", "log_dividends",
        "investment_activity", "sex", "race", "education",
        "marital stat", "class of worker", "major occupation code",
        "major industry code", "full or part time employment stat",
        "own business or self employed", "detailed industry recode",
        "detailed occupation recode", "veterans benefits",
    ]
    return df[cols]

# preprocessing
def preprocess(X, y):
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numerical_cols),
        ("cat", TargetEncoder(), categorical_cols),
    ])
    return preprocessor.fit_transform(X, y)

# apply PCA
def reduce_dim(X, variance=0.90):
    pca = PCA(n_components=variance, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X)
    print(f"PCA: {X.shape[1]} → {X_pca.shape[1]} dimensions ({variance*100:.0f}% variance)")
    return X_pca

# tune DBSCAN eps using k-distance plot
def plot_kdistance(X, min_samples):
    nn = NearestNeighbors(n_neighbors=min_samples, algorithm="ball_tree", n_jobs=-1)
    nn.fit(X)
    distances, _ = nn.kneighbors(X)
    k_dist = np.sort(distances[:, -1])

    plt.figure(figsize=(9, 4))
    plt.plot(k_dist, color="#4C72B0")
    plt.ylabel(f"Distance to {min_samples}th neighbour")
    plt.xlabel("Points sorted by distance")
    plt.title("K-Distance Plot — set EPS at the 'knee'")
    plt.tight_layout()
    plt.show()

    suggested = float(np.percentile(k_dist, 10))
    print(f"Suggested starting eps (10th percentile): {suggested:.4f}")
    return suggested

# run DBSCAN and evaluate
def run_dbscan(X, eps, min_samples):
    db = DBSCAN(eps=eps, min_samples=min_samples, algorithm="ball_tree", n_jobs=-1)
    labels = db.fit_predict(X)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()

    print(f"\nClusters found: {n_clusters}")
    print(f"Noise points: {n_noise} ({n_noise/len(labels)*100:.1f}%)")
    print(f"\nCluster sizes:\n{pd.Series(labels).value_counts().sort_index().to_string()}")

    core_mask = labels != -1
    if n_clusters >= 2 and core_mask.sum() > 1:
        sil = silhouette_score(
            X[core_mask], labels[core_mask],
            sample_size=min(10000, core_mask.sum()),
            random_state=RANDOM_STATE,
        )
        print(f"\nSilhouette score (non-noise): {sil:.4f}")
    else:
        print("\nNot enough clusters for silhouette — see tuning guide below")

    return labels

# cluster segment profiling
def profile_clusters(df_s, labels):
    df_s = df_s.copy()
    df_s["dbscan_cluster"] = labels

    num_cols = [c for c in ["age", "log_wage_per_hour", "log_dividends",
                             "investment_activity", "employment_stability",
                             "income_gt50k"] if c in df_s.columns]
    profile = df_s.groupby("dbscan_cluster")[num_cols].mean().round(3)

    for col in ["education", "major occupation code", "marital stat", "sex"]:
        if col in df_s.columns:
            profile[col] = df_s.groupby("dbscan_cluster")[col].agg(
                lambda x: x.mode().iloc[0]
            )

    profile["size"] = df_s.groupby("dbscan_cluster").size()
    profile["size_pct"] = (profile["size"] / len(df_s) * 100).round(1)

    print("\n── Cluster Profiles (−1 = noise/outliers) ───────────────────────")
    print(profile.to_string())
    return profile, df_s

# visualize clusters
def visualise(X_pca, labels, df_s, eps, min_samples):
    unique = sorted(set(labels))
    n_clusters = len(unique) - (1 if -1 in unique else 0)
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_clusters, 1)))

    fig, axes = plt.subplots(1, 1, figsize=(14, 5))

    # PCA scatter
    color_map = {}
    ci = 0
    for label in unique:
        if label == -1:
            color_map[label] = "lightgrey"
        else:
            color_map[label] = colors[ci]; ci += 1

    for label in unique:
        mask = labels == label
        name = f"Noise ({mask.sum()})" if label == -1 else f"Cluster {label} ({mask.sum()})"
        axes.scatter(X_pca[mask, 0], X_pca[mask, 1],
                        s=6, alpha=0.4, color=color_map[label], label=name)
    axes.set_xlabel("PC1"); axes.set_ylabel("PC2")
    axes.set_title(f"DBSCAN (eps={eps:.3f}, min_samples={min_samples})")
    axes.legend(markerscale=3, fontsize=8)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    DATA_PATH    = "../data/census-bureau.data"
    COLUMNS_PATH = "../data/census-bureau.columns"

    df = load_data(DATA_PATH, COLUMNS_PATH)
    df = clean_data(df)
    df = engineer_features(df)

    # Subsample
    np.random.seed(RANDOM_STATE)
    idx = np.random.choice(len(df), size=min(SAMPLE_SIZE, len(df)), replace=False)
    df_sample = df.iloc[idx].copy().reset_index(drop=True)
    print(f"Sample: {len(df_sample):,} records")

    X = select_features(df_sample)
    y = df_sample["income_gt50k"]
    X_proc = preprocess(X, y)
    X_pca  = reduce_dim(X_proc, variance=0.90)

    MIN_SAMPLES = max(5, X_pca.shape[1] * 2)
    suggested_eps = plot_kdistance(X_pca, MIN_SAMPLES)
    EPS = .4

    labels = run_dbscan(X_pca, eps=EPS, min_samples=MIN_SAMPLES)
    profile, df_sample = profile_clusters(df_sample, labels)
    visualise(X_pca, labels, df_sample, EPS, MIN_SAMPLES)


In [ ]:
# Hierarchical Clustering

RANDOM_STATE = 42
SAMPLE_SIZE  = 20000
N_CLUSTERS   = 5
DATA_PATH    = "../data/census-bureau.data"
COLUMNS_PATH = "../data/census-bureau.columns"

def load_data(data_path, columns_path):
    with open(columns_path) as f:
        columns = [line.strip() for line in f if line.strip()]
    df = pd.read_csv(data_path, names=columns)
    if 'hispanic origin' in df.columns:
        df['hispanic origin'] = df['hispanic origin'].fillna('Unknown')
    return df

def clean(df):
    df = df[df['age'] >= 18].copy()
    df = df.replace('?', 'Unknown')
    return df

def engineer_features(df):
    df['log_capital_gains'] = np.log1p(df['capital gains'])
    df['log_dividends']     = np.log1p(df['dividends from stocks'])
    df['has_investments']   = (
        (df['capital gains'] > 0) |
        (df['dividends from stocks'] > 0) |
        (df['capital losses'] > 0)
    ).astype(int)
    df['log_wage_per_hour'] = np.log1p(df['wage per hour'])
    return df

def select_features(df):
    features = [
        "age",
        "sex",
        "marital stat",
        "race",
        "weeks worked in year",
        "full or part time employment stat",
        "class of worker",
        "major occupation code",
        "major industry code",
        "num persons worked for employer",
        "education",
        "tax filer stat",
        "log_wage_per_hour",
        "log_capital_gains",
        "log_dividends",
        "has_investments",
    ]
    return df[features]

def preprocess(X, y):
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numerical_cols   = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numerical_cols),
        ("cat", TargetEncoder(random_state=RANDOM_STATE), categorical_cols)
    ])
    X_processed = preprocessor.fit_transform(X, y)
    return X_processed

# PCA for dimensionality reduction
def reduce(X, variance_threshold=0.9):
    pca = PCA(n_components=variance_threshold, svd_solver='full', random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X)
    print(f"PCA reduced to {X_pca.shape[1]} components (explained variance: {variance_threshold*100:.0f}%)")
    return X_pca

if __name__ == "__main__":

    print("Loading data...")
    df = load_data(DATA_PATH, COLUMNS_PATH)
    df = clean(df)
    df['label_binary'] = df['label'].apply(lambda x: 1 if '50000+' in x else 0)
    df = engineer_features(df)

    X = select_features(df)
    y = df['label_binary']

    print("Preprocessing...")
    X_processed = preprocess(X, y)

    print("Reducing dimensionality...")
    X_pca = reduce(X_processed, variance_threshold=0.9)

    # train on a sample for K clusters
    np.random.seed(RANDOM_STATE)
    sample_idx = np.random.choice(X_pca.shape[0], size=min(SAMPLE_SIZE, X_pca.shape[0]), replace=False)
    X_sample = X_pca[sample_idx]
    df_sample = df.iloc[sample_idx].copy().reset_index(drop=True)
    print(f"Sample size: {len(sample_idx):,}  |  PCA dims: {X_sample.shape[1]}")
    print(f"\n{'K':>4}  {'Silhouette':>12}  {'Calinski-Harabasz':>18}  {'Davies-Bouldin':>15}")
    print("-" * 55)

    results = []
    for k in range(2, 10):
        agglo = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = agglo.fit_predict(X_sample)
        sil = silhouette_score(X_sample, labels)
        ch = calinski_harabasz_score(X_sample, labels)
        db = davies_bouldin_score(X_sample, labels)
        results.append({"k": k, "silhouette": sil, "calinski_harabasz": ch, "davies_bouldin": db})
        print(f"{k:>4}  {sil:>12.4f}  {ch:>18.1f}  {db:>15.4f}")

    best_sil = max(results, key=lambda x: x["silhouette"])["k"]
    best_ch = max(results, key=lambda x: x["calinski_harabasz"])["k"]
    best_db = min(results, key=lambda x: x["davies_bouldin"])["k"]
    print(f"\nBest K for silhouette: {best_sil}  |  Calinski-Harabasz: {best_ch}  |  Davies-Bouldin: {best_db}")

    print(f"\nFitting final model with K={N_CLUSTERS}...")
    agglo_final = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward")
    hier_labels = agglo_final.fit_predict(X_sample)
    df_sample["hier_cluster"] = hier_labels

    # cluster profiling
    profile_cols = [c for c in ["age", "weeks worked in year", "num persons worked for employer", "label_binary"]
                    if c in df_sample.columns]

    num_profile = df_sample.groupby("hier_cluster")[profile_cols].mean().round(3)

    for col in ["education", "major occupation code", "marital stat", "sex"]:
        if col in df_sample.columns:
            num_profile[col] = df_sample.groupby("hier_cluster")[col].agg(
                lambda x: x.mode().iloc[0]
            )

    num_profile["size"] = df_sample.groupby("hier_cluster").size()
    num_profile["size_pct"] = (num_profile["size"] / len(df_sample) * 100).round(1)

    print("\n── Cluster Profiles ──────────────────────────────────────────────")
    print(num_profile.to_string())

    # metrics for chosen K
    chosen = next(r for r in results if r["k"] == N_CLUSTERS)
    print(f"\n── Metrics @ K={N_CLUSTERS} ──────────────────────────────────────")
    print(f"Silhouette score: {chosen['silhouette']:.4f}")
    print(f"Calinski-Harabasz score: {chosen['calinski_harabasz']:.1f}")
    print(f"Davies-Bouldin score: {chosen['davies_bouldin']:.4f}")